# Episode 3: Reality check of the bronze tables

In episode 2 we created the bronze tables **straight from the files**, and Spark guessed the column types on its own (schema-on-read).
Now we check what it actually guessed and where it got it wrong.

What we expect to find:
- **CSV has no types.** Without `inferSchema` every column is a `STRING`, including dates, prices and quantities.
- **JSON does not recognise dates.** A timestamp in JSON is just text, so it lands as a `STRING`.
- **JSON numbers with a fraction become `DOUBLE`.** For money that is a bad choice: `0.1 + 0.2 != 0.3`.
- **Locale-specific formats** (`1,299.99`, `03/14/2025 07:22:33 PM`) that no automatic cast will understand.

Plan:
1. Overview: which types ended up in which table.
2. Save the current schemas as DDL files (a snapshot we can compare against later).
3. Find `STRING` columns that look like numbers, dates or booleans.
4. Dig into the suspicious columns.
5. Write down the findings - they become the rules for the silver layer.

In [0]:
CATALOG_PREFIX = "moustache_markt"
ENV = "dev"
TARGET = f"{CATALOG_PREFIX}_{ENV}"

# All bronze tables, found the same way as in the previous notebooks
TABLES = [
    f"{TARGET}.{row.table_schema}.{row.table_name}"
    for row in spark.sql(
        """
        SELECT table_schema, table_name
        FROM system.information_schema.tables
        WHERE table_catalog = :catalog AND startswith(table_schema, 'bronze_')
        ORDER BY table_schema, table_name
        """,
        args={"catalog": TARGET},
    ).collect()
]
TABLES

## 1. Overview: types per table

`information_schema.columns` has one row per column, for every table at once - no need to open the tables one by one.
The pivot gives one row per table and one column per data type: a table made of nothing but `STRING` stands out immediately.

In [0]:
columns_df = spark.sql(
    """
    SELECT table_schema, table_name, ordinal_position, column_name, data_type, full_data_type
    FROM system.information_schema.columns
    WHERE table_catalog = :catalog AND startswith(table_schema, 'bronze_')
    ORDER BY table_schema, table_name, ordinal_position
    """,
    args={"catalog": TARGET},
)

# How many columns of each type in every table
display(columns_df.groupBy("table_schema", "table_name").pivot("data_type").count().orderBy("table_schema", "table_name"))

# And the full list, column by column
display(columns_df)

## 2. Save the schema as DDL

`SHOW CREATE TABLE` returns the exact statement that would recreate the table.
We save one `.sql` file per table into `schemas/` in the repo. Committed to Git, it becomes a **schema snapshot**:
when a source system changes its format, `git diff` shows exactly which column changed.

In [0]:
import os

DDL_DIR = os.path.join(os.getcwd(), "schemas")   # notebook folder = repo root
os.makedirs(DDL_DIR, exist_ok=True)

for table_name in TABLES:
    ddl = spark.sql(f"SHOW CREATE TABLE {table_name}").first()[0]
    file_name = table_name.split(".", 1)[1] + ".sql"   # bronze_pos.receipt_line.sql
    with open(os.path.join(DDL_DIR, file_name), "w") as f:
        f.write(ddl + "\n")
    print(f"schemas/{file_name}")

In [0]:
DDL_DIR

In [0]:
# One of them, to see what such a file looks like
print(open(os.path.join(DDL_DIR, "bronze_pos.receipt_line.sql")).read())

## 3. STRING columns that are something else

For every `STRING` column we count how many values **can be converted** to another type.

`try_cast` is a cast that returns `NULL` instead of failing (a plain `CAST` throws an error on serverless, where ANSI mode is on).
So `count(try_cast(x AS BIGINT))` = how many values are valid integers.

How to read the result:
- **`suggestion`**: 100% of the values fit this type.
- **`almost`**: 90-99.9% fit. These are the interesting ones: a few values in a different format, e.g. `1,299.99` among `199.99`.
- **0% everywhere + examples that look like dates or numbers**: a format that no cast understands (`03/14/2025 07:22:33 PM`).

Do not trust the suggestion blindly:
- codes with leading zeros (EAN, postal code) pass as `int`, but converting them loses the zeros,
- a phone number passes as `int`, but it is not a number you would add up,
- `try_cast('1' AS BOOLEAN)` is `true`, so 0/1 columns also pass as `boolean`.

In [0]:
import pandas as pd
from pyspark.sql import functions as F

# Each check returns NULL when the value does NOT fit the type. {c} = column name.
CHECKS = {
    "int": "try_cast({c} AS BIGINT)",
    "decimal": "try_cast({c} AS DECIMAL(38, 6))",
    "date": "CASE WHEN length({c}) = 10 THEN try_cast({c} AS DATE) END",   # only yyyy-MM-dd, otherwise every timestamp passes too
    "timestamp": "try_cast({c} AS TIMESTAMP)",
    "boolean": "try_cast({c} AS BOOLEAN)",
}


def string_column_report(table_name):
    df = spark.table(table_name)
    # Our own technical columns (_source_file, _ingested_at) are skipped
    string_columns = [f.name for f in df.schema.fields if f.dataType.typeName() == "string" and not f.name.startswith("_")]

    # All counts in ONE pass over the table: filled values + values that fit each type, per column
    aggregations = []
    for c in string_columns:
        aggregations.append(F.count(c).alias(f"{c}__filled"))
        for check, sql in CHECKS.items():
            aggregations.append(F.count(F.expr(sql.format(c=f"`{c}`"))).alias(f"{c}__{check}"))
    counts = df.agg(*aggregations).first()
    examples = df.limit(3).collect()

    rows = []
    for c in string_columns:
        filled = counts[f"{c}__filled"]
        share = {check: counts[f"{c}__{check}"] / filled if filled else 0 for check in CHECKS}
        rows.append({
            "column": c,
            "filled": filled,
            **{f"{check} %": round(100 * share[check], 1) for check in CHECKS},
            "suggestion": next((check for check in CHECKS if filled and share[check] == 1), ""),
            "almost": next((check for check in CHECKS if 0.9 <= share[check] < 1), ""),
            "examples": [row[c] for row in examples],
        })
    return pd.DataFrame(rows)


for table_name in TABLES:
    print(table_name)
    display(string_column_report(table_name))

## 4. Dig into a suspicious column

`bad_values` shows the values that do **not** fit a given type, most frequent first.
This is where you find out *why* a column is only 97% decimal, or why a date column is 0% timestamp.

In [0]:
def bad_values(table_name, column, target_type, limit=20):
    return spark.sql(f"""
        SELECT `{column}` AS value, count(*) AS rows
        FROM {table_name}
        WHERE `{column}` IS NOT NULL AND try_cast(`{column}` AS {target_type}) IS NULL
        GROUP BY 1
        ORDER BY rows DESC
        LIMIT {limit}
    """)


# Examples - replace with whatever the report above flagged
display(bad_values(f"{TARGET}.bronze_pos.receipt_line", "item_price", "DECIMAL(38, 6)"))
display(bad_values(f"{TARGET}.bronze_pos.receipt_line", "receipt_local_ts", "TIMESTAMP"))

## 5. Numbers stored as DOUBLE

JSON has a single number type, so Spark reads every number with a fraction as `DOUBLE` (a binary floating point).
For prices, amounts and tax that is a bug waiting to happen: sums stop adding up to the cent.
Money belongs in `DECIMAL`.

In [0]:
display(columns_df.where("data_type IN ('DOUBLE', 'FLOAT')"))

## 6. Nested columns

The report in step 3 only looks at top-level columns. `STRUCT` and `ARRAY` columns (e-commerce orders: `items`, `delivery`)
can hide the same problems one level deeper, e.g. a timestamp as a string inside `delivery`.
`printSchema` shows the whole tree.

In [0]:
nested_tables = sorted({
    f"{TARGET}.{row.table_schema}.{row.table_name}"
    for row in columns_df.where("data_type IN ('STRUCT', 'ARRAY', 'MAP')").collect()
})

for table_name in nested_tables:
    print(table_name)
    spark.table(table_name).printSchema()

## 7. Findings

Fill this in as you go through the report. Every row is a rule for the silver layer.

| Table | Column | Bronze type | Should be | Why / how to convert |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |